# Lab 06 : Diffusion Model for MNIST generation - exercise

Goal:The primary objective is to build and train a generative diffusion model that can synthesize new, realistic images of MNIST digits by learning to reverse a multi-step noise-addition process.

Implement:

Noise Schedules: It defines linear $\beta$ schedules and calculates derived coefficients like $\alpha_t$ and $\bar{\alpha}_t$ for the forward diffusion process.

Architecture: It utilizes a U-Net backbone with downsampling/upsampling layers and Positional Encodings to handle time-step information.

Training Pipeline: The notebook implements a forward pass that adds noise to clean images and a backward pass where the network learns to predict and remove that noise using Mean Squared Error (MSE) loss.

Generation: It features a generative loop that starts from pure Gaussian noise and iteratively denoises it over 1,000 steps to produce clear digit images.

In [ ]:
# For Google Colaboratory
import sys, os
if 'google.colab' in sys.modules:
    # mount google drive
    from google.colab import drive
    drive.mount('/content/gdrive')
    path_to_file = '/content/gdrive/My Drive/IPAM_Mar26_codes/labs_lecture06/lab06_dm_mnist'
    print(path_to_file)
    # move to Google Drive directory
    os.chdir(path_to_file)
    !pwd


In [ ]:
# Libraries
import torch
import torch.nn as nn
import torch.optim as optim
import time
#import utils
import matplotlib.pyplot as plt
import logging
logging.getLogger().setLevel(logging.CRITICAL) # remove warnings
import os, datetime

# PyTorch version and GPU
print(torch.__version__)
if torch.cuda.is_available():
  print(torch.cuda.get_device_name(0))
  device= torch.device("cuda") # use GPU
else:
  device= torch.device("cpu")
print(device)


In [ ]:
# Import MNIST if necessary
import os.path
def check_mnist_dataset_exists(path_data='data/'):
    flag_train_data = os.path.isfile(path_data + 'mnist/train_data.pt')
    flag_train_label = os.path.isfile(path_data + 'mnist/train_label.pt')
    flag_test_data = os.path.isfile(path_data + 'mnist/test_data.pt')
    flag_test_label = os.path.isfile(path_data + 'mnist/test_label.pt')
    if flag_train_data==False or flag_train_label==False or flag_test_data==False or flag_test_label==False:
        print('MNIST dataset missing - downloading...')
        import torchvision
        import torchvision.transforms as transforms
        trainset = torchvision.datasets.MNIST(root=path_data + 'mnist/temp', train=True,
                                                download=True, transform=transforms.ToTensor())
        testset = torchvision.datasets.MNIST(root=path_data + 'mnist/temp', train=False,
                                               download=True, transform=transforms.ToTensor())
        train_data=torch.Tensor(60000,28,28)
        train_label=torch.LongTensor(60000)
        for idx , example in enumerate(trainset):
            train_data[idx]=example[0].squeeze()
            train_label[idx]=example[1]
        torch.save(train_data,path_data + 'mnist/train_data.pt')
        torch.save(train_label,path_data + 'mnist/train_label.pt')
        test_data=torch.Tensor(10000,28,28)
        test_label=torch.LongTensor(10000)
        for idx , example in enumerate(testset):
            test_data[idx]=example[0].squeeze()
            test_label[idx]=example[1]
        torch.save(test_data,path_data + 'mnist/test_data.pt')
        torch.save(test_label,path_data + 'mnist/test_label.pt')
    return path_data
data_path=check_mnist_dataset_exists()
train_data=torch.load(data_path+'mnist/train_data.pt')
train_label=torch.load(data_path+'mnist/train_label.pt')
print(train_data.size())


## Diffusion model schedules

In [ ]:
# Global constants
d = 48 # latent dimension
dPE = 128 # PE dimension
beta_1 = 0.0001 # for denoising schedule in DM
beta_T = 0.02 # for denoising schedule in DM
num_t = 150 # number of time steps in diffusion process
num_t = 1000 # DEBUG
print('beta_1, beta_T, num_t, d, dPE:',beta_1,beta_T,num_t,d,dPE)


In [ ]:
# Plot diffusion model coefficients w.r.t. time step t

#############################
# First plot : alpha_t and alpha_bar_t
#############################
# Compute the Noise Schedule
alpha_t = 1.0 - torch.linspace(beta_1, beta_T, num_t).to('cpu') 
alpha_bar_t = torch.cumprod(alpha_t, dim=0)
# Convert to numpy for plotting
steps = range(num_t)
alpha_t_np = alpha_t.numpy()
alpha_bar_t_np = alpha_bar_t.numpy()

#############################
# Second plot : sqrt_alpha_bar_t and sqrt_one_minus_alpha_bar_t
#############################
# Compute the Noise Schedule
alpha_t = 1.0 - torch.linspace(beta_1, beta_T, num_t).to('cpu') 
alpha_bar_t = torch.cumprod(alpha_t, dim=0)
# Compute your new Square Root terms
sqrt_alpha_bar_t = alpha_bar_t.sqrt()
sqrt_one_minus_alpha_bar_t = (1.0 - alpha_bar_t).sqrt()
# Convert to numpy for plotting
steps = range(num_t)
s_alpha = sqrt_alpha_bar_t.numpy()
s_one_minus = sqrt_one_minus_alpha_bar_t.numpy()

#############################
# Figures
#############################
fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(9, 9))
# Top Figure: Noise Schedule
color_alpha = 'tab:blue'
ax_top.set_ylabel(r'$\alpha_t$', color=color_alpha)
lns1 = ax_top.plot(steps, alpha_t_np, color=color_alpha, label=r'$\alpha_t$ (Step Variance)')
ax_top.tick_params(axis='y', labelcolor=color_alpha)
ax_top.grid(True, linestyle='--', alpha=0.6)
# Add the secondary axis for the top plot
ax_top_twin = ax_top.twinx()
color_bar = 'tab:orange'
ax_top_twin.set_ylabel(r'$\bar{\alpha}_t$', color=color_bar)
lns2 = ax_top_twin.plot(steps, alpha_bar_t_np, color=color_bar, linewidth=2, label=r'$\bar{\alpha}_t$ (Cumulative Product)')
ax_top_twin.tick_params(axis='y', labelcolor=color_bar)
ax_top.set_title(r'Diffusion Model Noise Schedule: $\alpha_t$ and $\bar{\alpha}_t$')
# Combine legends from both axes into one box for the top plot
lns_top = lns1 + lns2
labs_top = [l.get_label() for l in lns_top]
ax_top.legend(lns_top, labs_top, loc='center right')
# Bottom Figure: Signal vs. Noise Scaling 
ax_bot.plot(steps, s_alpha, label=r'$\sqrt{\bar{\alpha}_t}$ (Signal Scale)', color='tab:green', linewidth=2)
ax_bot.plot(steps, s_one_minus, label=r'$\sqrt{1 - \bar{\alpha}_t}$ (Noise Scale)', color='tab:red', linewidth=2)
ax_bot.set_xlabel('Timestep (t)')
ax_bot.set_ylabel('Scaling Factor Value')
ax_bot.set_title('Diffusion Forward Process: Signal vs. Noise Scaling')
ax_bot.grid(True, linestyle='--', alpha=0.6)
ax_bot.legend(loc='center right')
# Adjust layout to prevent overlap
fig.tight_layout()
plt.show()


from IPython.display import display, Latex
latex_str = r'$$\Huge \textrm{with } \bar{\alpha}_t = \prod_{i=0}^t \alpha_i$$'
display(Latex(latex_str))

from IPython.display import display, Latex
latex_str = r'$$\Huge x_t = \sqrt{\bar{\alpha}_t} \cdot x_0 + \sqrt{1 - \bar{\alpha}_t} \cdot \epsilon$$ '
display(Latex(latex_str))



## Design one forward pass and the generative process

1. Forward diffusion process : Jump from $x_0$ to $x_t$ in one step:
\begin{eqnarray*}
    x_t = \sqrt{\bar{\alpha_t}}x_0+\sqrt{1-\bar{\alpha_t}}\epsilon_0, \textrm{ where } \epsilon_0\sim\mathcal{N}(0, I) \quad (1)
\end{eqnarray*}

2. Generation process :
- Randomly sample from $x_{T-1}\sim\mathcal{N}(0, I)$.
- Generate $x_{t-1}$ given $x_t$ following the equation
\begin{eqnarray*}
    &&x_{t-1} = \mu_t + \sigma_t z, \quad \textrm{where } z\sim\mathcal{N}(0, I), \quad (2) \\
    &&\mu_t = \frac{1}{\sqrt{\alpha_t}}(x_t-\frac{1-\alpha_t}{\sqrt{1-\bar{\alpha_t}}}
    \epsilon_\theta(x_t, t)), \quad (3)\\
    &&\sigma^2_t = \frac{(1-\alpha_t)(1-\bar{\alpha}_{t-1})}{1-\bar{\alpha_t}}. \quad (4)\\
\end{eqnarray*}
We recurrently calculate $x_{t-1}$ until $x_0$.




In [ ]:
# Global constants
n = train_data.size(1) # num of pixels along one spatial dimension
bs = 100 # 256 # bs : batch size
N = train_data.size(0) # num of training data
print('n, bs, N:',n,bs,N)


In [ ]:
# Network design
class two_conv_layers(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_dim, out_dim, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(in_dim)
        self.conv2 = nn.Conv2d(out_dim, out_dim, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_dim)
        self.linear_pe = nn.Linear(dPE, out_dim, bias=True)
    def forward(self, h, p, first_rc=False):
      if first_rc:
          h = torch.relu(self.conv1(self.bn1(h))) + torch.relu(self.linear_pe(p).unsqueeze(2).unsqueeze(3))
      else:
          h = h + torch.relu(self.conv1(self.bn1(h))) + torch.relu(self.linear_pe(p).unsqueeze(2).unsqueeze(3))
      h = h + torch.relu(self.conv2(self.bn2(h)))
      return h

class downsample_layer(nn.Module):
    def __init__(self, in_dim, out_dim, padding):
        super().__init__()
        self.bn = nn.BatchNorm2d(in_dim)
        self.conv_pool = nn.Conv2d(in_dim, out_dim, kernel_size=3, stride=2, padding=padding)
        self.conv_block = two_conv_layers(out_dim, out_dim) # in_dim=out_dim
    def forward(self, h, p):
      h = torch.relu(self.conv_pool(self.bn(h))) # downsampling with conv pooling
      h = self.conv_block(h, p)
      return h

class upsample_layer(nn.Module):
    def __init__(self, in_dim, out_dim, output_padding):
        super().__init__()
        self.bn = nn.BatchNorm2d(in_dim)
        self.deconv = nn.ConvTranspose2d(in_dim, out_dim, kernel_size=3, stride=2, padding=1, output_padding=output_padding)
        self.conv_block = two_conv_layers(2*out_dim, out_dim)
    def forward(self, h_level, h_level_minus_one, p):
        h = torch.cat( ( torch.relu(self.deconv(self.bn(h_level))), h_level_minus_one ), dim=1 )
        h = self.conv_block(h, p, True)
        return h

def generate_positional_encoding(max_t, dim_pe):
    pe = torch.zeros(max_t, dim_pe)
    position = torch.arange(0, max_t, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dim_pe, 2).float() * (-torch.log(torch.tensor(10000.0)) / dim_pe))
    pe[:,0::2] = torch.sin(position * div_term) # all t, d=0,2,4,..
    pe[:,1::2] = torch.cos(position * div_term) # all t, d=1,3,5,..
    return pe # [max_t, dim_pe]

# Define UNet architecture for image denoising
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_block1 = two_conv_layers(  1,    d)
        self.downsample1 = downsample_layer( d,  2*d, 1)
        self.downsample2 = downsample_layer(2*d, 4*d, 1)
        self.downsample3 = downsample_layer(4*d, 8*d, 1)
        self.upsample1   = upsample_layer(  8*d, 4*d, 0)
        self.upsample2   = upsample_layer(  4*d, 2*d, 1)
        self.upsample3   = upsample_layer(  2*d,   d, 1)
        self.bn          = nn.BatchNorm2d(    d)
        self.conv        = nn.Conv2d(         d,   1, kernel_size=1)
        self.pe          = generate_positional_encoding(num_t, dPE)
        self.mlp_pe = nn.Sequential(nn.Embedding(num_t, dPE), nn.ReLU(), nn.Linear(dPE, dPE, bias=True))
    def forward(self, h_t, sample_t):
        h = h_t.unsqueeze(1)                     # [bs, 1, n, n]
        p = self.mlp_pe(sample_t)                # [bs, 128]
        h1_down = self.conv_block1(h, p, True)   # [bs, 64, 28, 28]
        h2_down = self.downsample1(h1_down, p)      # [bs, 128, 14, 14]
        h3_down = self.downsample2(h2_down, p)      # [bs, 256, 7, 7]
        h4_down = self.downsample3(h3_down, p)      # [bs, 512, 4, 4]
        h3_up = self.upsample1(h4_down, h3_down, p) # [bs, 256, 7, 7]
        h2_up = self.upsample2(h3_up, h2_down, p)   # [bs, 128, 14, 14]
        h1_up = self.upsample3(h2_up, h1_down, p)   # [bs, 64, 28, 28]
        h = self.conv(self.bn(h1_up))            # [bs, 1, 28, 28]
        h = h.squeeze()                          # [bs, 28, 28]
        return h

# Define DDPM architecture
class DDPM(nn.Module):

    def __init__(self, num_t, beta_1, beta_T):
        super().__init__()
        self.num_t = num_t
        self.alpha_t = 1.0 - torch.linspace(beta_1, beta_T, num_t).to(device) # [num_t]
        self.alpha_bar_t = torch.cumprod( self.alpha_t, dim=0) # [num_t]
        self.UNet = UNet()

    def forward_process(self, x0, sample_t, eps0): # add noise

        # Noisy image at time step t is given by Eq.(1)
        # Hint: You may use ".sqrt()" for square root and ".view(bs,1,1)" to reshape the schedulers
        # COMPLETE HERE
        sqrt_alpha_bar_t =    # [bs]
        sqrt_one_minus_alpha_bar_t =    # [bs]
        x_t =    # [bs, n, n]
        # COMPLETE HERE
        return x_t

    def backward_process(self, x_t, sample_t): # denoise
        # UNet is our denoising architecture 
        # Inputs : x_t (image noised with Eq.(1)), size=[batch_size, n, n], n is the number of pixels along one dimension
        #          sample_t (random time step between 0 and T-1=999 if e.g. T=1000), size=[batch_size]
        # Output : x_t_minus_one (denoised image), size=[batch_size, n, n]
        # COMPLETE HERE
        x_t_minus_one =     # [bs, 28, 28]
        # COMPLETE HERE
        return x_t_minus_one

    def generate_process_ddpm(self, num_gen_images):

        # Start from pure noise at t=T-1
        # Hint : You may use "torch.randn()" to create a tensor with Normal distribution and ".to(device)" to transfer to gpu if available 
        # COMPLETE HERE
        x_t =     # x_{T-1}
        # COMPLETE HERE
        
        for t_idx in range(self.num_t-1, -1, -1): # range(start, stop, step), decrease time steps from t=T-2, T-3, ..., 1, 0=t_final(clean image)

            # batch of time step t=T-2, T-3, ..., 1, 0, size=(num_gen_images)
            # Hint : You may use "torch.ones(N) * t" to produce a vector of N "t" values and ".long()" to convert to integer type
            # COMPLETE HERE
            batch_t = (torch.ones(num_gen_images) * t_idx).long().to(device)
            # COMPLETE HERE
            
            # Predict the noise using the backward/denoising process with UNet
            # COMPLETE HERE
            noise_pred_t =  
            # COMPLETE HERE
            
            # Compute mean_t Eq.(3)
            c1_t = 1.0 / self.alpha_t[t_idx].sqrt()
            c2_t = (1.0 - self.alpha_t[t_idx]) / ( 1.0 - self.alpha_bar_t[t_idx] ).sqrt()
            mean_t = c1_t * ( x_t - c2_t * noise_pred_t )  # Eq.(3)

            # Compute x_{t-1} Eq.(2)
            # Add noise for all time steps EXCEPT the last one (t=0) as
            # we do not add the noise term (\sigma_t z) because the denoising process is finished.
            if t_idx > 0:
                z = torch.randn_like(x_t)
                sigma_t_sqr = (1.0 - self.alpha_t[t_idx]) * ( 1.0 - self.alpha_bar_t[t_idx-1]) / ( 1.0 - self.alpha_bar_t[t_idx]) # Eq.(4)
                # COMPLETE HERE
                x_t_minus_one =      # Eq.(2)
                # COMPLETE HERE
            else:
                x_t_minus_one = mean_t # Eq.(2) with z=0 at t=0

            x_t = x_t_minus_one
            
        x0 = x_t # last time step provides the clean generated image x_{t=0}
        
        return x0


# Instantiate the network
net = DDPM(num_t, beta_1, beta_T)
net = net.to(device)
def display_num_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    print('Number of parameters: {} ({:.2f} million)'.format(nb_param, nb_param/1e6))
display_num_param(net)


# Test the forward pass, backward pass and gradient update with a single batch
init_lr = 0.001
optimizer = torch.optim.Adam(net.parameters(), lr=init_lr)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.95, patience=1)
idx_images = torch.LongTensor(bs).random_(0,N)
batch_x0 = train_data[idx_images,:,:].to(device) # [bs, n, n]

# Sample a batch of time steps between 0 and T-1=999 if e.g. T=1000, size(batch_sample_t) = [bs=batch_size]
# Hints : You may use "torch.randint(low=0, high, size)" that returns a tensor filled w/ random integers generated uniformly between low and high-1
#         as well as ".long()" to convert to integer type and ").to(device)" to transfer to gpu if available 
# COMPLETE HERE
batch_sample_t =     # random interger in {0,1,...,T-1} [bs]
# COMPLETE HERE

print('batch_sample_t',batch_sample_t.size())
batch_noise_t = torch.randn(batch_x0.size()).to(device) # [bs, n, n]
x_t = net.forward_process(batch_x0, batch_sample_t, batch_noise_t) # [bs, n, n]
print('x_t',x_t.size())
noise_pred_t = net.backward_process(x_t, batch_sample_t) # [bs, n, n]
print('noise_pred_t',noise_pred_t.size())
loss_DDPM = torch.nn.MSELoss()(noise_pred_t, batch_noise_t)
loss = loss_DDPM
optimizer.zero_grad()
loss.backward()
optimizer.step()
with torch.no_grad():
  batch_x_0 = net.generate_process_ddpm(4)
  print('batch_x_0',batch_x_0.size())


In [ ]:
# Visualize generated images with DDPM
def visualize_generated_images(net, num_generated_images=16):
    net.eval()
    with torch.no_grad():
      # num_generated_images = 16
      batch_x_0 = net.generate_process_ddpm(num_generated_images)
      print('batch_x_0',batch_x_0.size())
      x_hat = batch_x_0.squeeze().detach().to('cpu')
    figure, axis = plt.subplots(4, 4)
    figure.set_size_inches(10,10)
    i,j,cpt=0,0,0; axis[i,j].imshow(x_hat[cpt,:,:], cmap='gray'); axis[i,j].set_title("Generated w/ DDPM"); axis[i,j].axis('off')
    i,j,cpt=1,0,1; axis[i,j].imshow(x_hat[cpt,:,:], cmap='gray'); axis[i,j].set_title("Generated w/ DDPM"); axis[i,j].axis('off')
    i,j,cpt=2,0,2; axis[i,j].imshow(x_hat[cpt,:,:], cmap='gray'); axis[i,j].set_title("Generated w/ DDPM"); axis[i,j].axis('off')
    i,j,cpt=3,0,3; axis[i,j].imshow(x_hat[cpt,:,:], cmap='gray'); axis[i,j].set_title("Generated w/ DDPM"); axis[i,j].axis('off')
    i,j,cpt=0,1+0,4; axis[i,j].imshow(x_hat[cpt,:,:], cmap='gray'); axis[i,j].set_title("Generated w/ DDPM"); axis[i,j].axis('off')
    i,j,cpt=1,1+0,5; axis[i,j].imshow(x_hat[cpt,:,:], cmap='gray'); axis[i,j].set_title("Generated w/ DDPM"); axis[i,j].axis('off')
    i,j,cpt=2,1+0,6; axis[i,j].imshow(x_hat[cpt,:,:], cmap='gray'); axis[i,j].set_title("Generated w/ DDPM"); axis[i,j].axis('off')
    i,j,cpt=3,1+0,7; axis[i,j].imshow(x_hat[cpt,:,:], cmap='gray'); axis[i,j].set_title("Generated w/ DDPM"); axis[i,j].axis('off')
    i,j,cpt=0,2+0,8; axis[i,j].imshow(x_hat[cpt,:,:], cmap='gray'); axis[i,j].set_title("Generated w/ DDPM"); axis[i,j].axis('off')
    i,j,cpt=1,2+0,9; axis[i,j].imshow(x_hat[cpt,:,:], cmap='gray'); axis[i,j].set_title("Generated w/ DDPM"); axis[i,j].axis('off')
    i,j,cpt=2,2+0,10; axis[i,j].imshow(x_hat[cpt,:,:], cmap='gray'); axis[i,j].set_title("Generated w/ DDPM"); axis[i,j].axis('off')
    i,j,cpt=3,2+0,11; axis[i,j].imshow(x_hat[cpt,:,:], cmap='gray'); axis[i,j].set_title("Generated w/ DDPM"); axis[i,j].axis('off')
    i,j,cpt=0,3+0,12; axis[i,j].imshow(x_hat[cpt,:,:], cmap='gray'); axis[i,j].set_title("Generated w/ DDPM"); axis[i,j].axis('off')
    i,j,cpt=1,3+0,13; axis[i,j].imshow(x_hat[cpt,:,:], cmap='gray'); axis[i,j].set_title("Generated w/ DDPM"); axis[i,j].axis('off')
    i,j,cpt=2,3+0,14; axis[i,j].imshow(x_hat[cpt,:,:], cmap='gray'); axis[i,j].set_title("Generated w/ DDPM"); axis[i,j].axis('off')
    i,j,cpt=3,3+0,15; axis[i,j].imshow(x_hat[cpt,:,:], cmap='gray'); axis[i,j].set_title("Generated w/ DDPM"); axis[i,j].axis('off')
    plt.show()
    net.train()
    

In [ ]:
## Training loop
del net
net = DDPM(num_t, beta_1, beta_T)
net = net.to(device)
display_num_param(net)

# Optimizer
init_lr = 0.0003
optimizer = torch.optim.Adam(net.parameters(), lr=init_lr)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.95, patience=1)

# Number of epochs
nb_epochs = 20

# Saving checkpoint
time_stamp = datetime.datetime.now().strftime("%y-%m-%d--%H-%M-%S")
checkpoint_dir = os.path.join("checkpoint")
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)
epoch_ckpt = 0
tot_time_ckpt = 0

start=time.time()
for epoch in range(nb_epochs):

    running_loss = 0.0
    num_batches = 0

    shuffled_indices = torch.randperm(60000)

    for count in range(0,60000,bs):

        idx_images = shuffled_indices[count : count+bs]
        batch_x0 = train_data[idx_images,:,:].to(device) # [bs, n, n]
        batch_sample_t = torch.randint(0, num_t, (bs,)).long().to(device) # [bs]
        batch_noise_t = torch.randn(batch_x0.size()).to(device) # [bs, n, n]
        x_t = net.forward_process(batch_x0, batch_sample_t, batch_noise_t) # [bs, n, n]
        noise_pred_t = net.backward_process(x_t, batch_sample_t) # [bs, n, n]
        loss_DDPM = torch.nn.MSELoss()(noise_pred_t, batch_noise_t)
        loss = loss_DDPM
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # COMPUTE STATS
        running_loss += loss.detach().item()
        num_batches += 1

    # AVERAGE STATS THEN DISPLAY
    total_loss = running_loss/num_batches
    scheduler.step(total_loss)
    elapsed = (time.time()-start)/60
    print('epoch=',epoch_ckpt+epoch, '\t time=', tot_time_ckpt+elapsed,'min', '\t lr=', optimizer.param_groups[0]['lr']  ,'\t loss=', total_loss )

    # Visualize 
    if not epoch%10:
        visualize_generated_images(net, num_generated_images=16)
    
    # Saving checkpoint
    torch.save({
        'epoch': epoch,
        'tot_time': elapsed,
        'loss': total_loss,
        'net': net.state_dict(),
        'optimizer': optimizer.state_dict(),
        }, '{}.pkl'.format(checkpoint_dir + "/checkpoint_" + time_stamp ))

    # Check lr value
    if optimizer.param_groups[0]['lr'] < 2*10**-4: # 2*10**-4: quick, # 10**-6: slow
      print("\n lr is equal to min lr -- training stopped\n")
      break


In [ ]:
# Loading pre-trained net
net = DDPM(num_t, beta_1, beta_T)
net = net.to(device)
display_num_param(net)
checkpoint_file = "checkpoint/dm_mnist_checkpoint_26-02-17--12-24-44.pkl"
print(f'Loading pre-trained model {checkpoint_file}')
if not os.path.exists(checkpoint_file):
    print(f'Downloading {checkpoint_file} ...')
    os.makedirs('checkpoint', exist_ok=True)
    dropbox_url = "https://www.dropbox.com/scl/fi/e8k4qjp8n7no78ivmxtx9/dm_mnist_checkpoint_26-02-17-12-24-44.pkl?rlkey=6r58t1zgw91ustn0v327icu2o&dl=1"
    !wget -q "{dropbox_url}" -O "{checkpoint_file}" 
checkpoint = torch.load(checkpoint_file, map_location=device)
net.load_state_dict(checkpoint['net']) 
visualize_generated_images(net, num_generated_images=16)
